In [7]:
!pip install sqlalchemy pymysql

Defaulting to user installation because normal site-packages is not writeable
  Using cached pymysql-1.1.3-py3-none-any.whl.metadata (4.3 kB)
Using cached pymysql-1.1.3-py3-none-any.whl (45 kB)


Imports

In [4]:

from sqlalchemy import create_engine, String, select, func, DateTime, Date
import pymysql
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, sessionmaker
import datetime 
from sqlalchemy.sql import func 
from sqlalchemy.dialects.mysql import LONGTEXT, TINYINT, DOUBLE
import enum

Setup generale

In [5]:
USERNAME = 'leonardo' 
PASSWORD = 'flandia' #security first...
HOST = 'localhost'
PORT = '3306'
DATABASE = 'mydb'

connADB = f"mysql+pymysql://{USERNAME}:{PASSWORD}@{HOST}:{PORT}/{DATABASE}" #connessione
motore = create_engine(connADB, echo = True) #echo = True per debugging

Classi

In [7]:


class Base(DeclarativeBase): #classe base
    pass

class TipoENUM(enum.Enum): #classe per definite l'ENUM
    org = "org"
    pers = "pers"


class Persona(Base):
    __tablename__ = "Persona"
    id: Mapped[str] = mapped_column("ID persona",String(25), primary_key=True)
    nome: Mapped[str] = mapped_column(String(20))
    cognome: Mapped[str] = mapped_column(String(20))
    secondonome: Mapped[str] = mapped_column("Secondo nome",String(45))
    titolo: Mapped[str] = mapped_column(String(45))
    sesso: Mapped[str] = mapped_column(String(1))
    Stato: Mapped[str] = mapped_column("Stato di residenza",String(30))
    Comune: Mapped[str] = mapped_column(String(45))
    Via: Mapped[str] = mapped_column(String(45))
    cap: Mapped[int] = mapped_column("CAP")
    Ruolo: Mapped[str] = mapped_column("Tipo/Ruolo",String(45))
    Lavora: Mapped[str] = mapped_column("Lavora per",String(30))

class Organizzazione(Base):
    __tablename__ = "Organizzazione"
    id: Mapped[str] = mapped_column("ID Organizzazione",String(30), primary_key=True)
    nome: Mapped[str] = mapped_column(String(20))
    proprietario: Mapped[str] = mapped_column("ID Proprietario",String(30))
    Stato: Mapped[str] = mapped_column("Paese",String(30))
    Comune: Mapped[str] = mapped_column(String(45))
    Via: Mapped[str] = mapped_column("Sede (indirizzo)",String(45))
    cap: Mapped[int] = mapped_column("CAP")
    Ruolo: Mapped[str] = mapped_column("Tipo/Ruolo",String(45))
    Lavora: Mapped[str] = mapped_column("Lavora per",String(30))

class File(Base):
    __tablename__ = "File"
    id: Mapped[str] = mapped_column("File ID",String(20), primary_key=True)
    link: Mapped[str] = mapped_column("Link",String(100))
    data: Mapped[datetime.date] = mapped_column(Date)
    Batch: Mapped[int] = mapped_column("Batch",TINYINT)
    Redacted: Mapped[int] = mapped_column("Redacted %",TINYINT)
    descrizione: Mapped[str] = mapped_column(String(400))
    size: Mapped[int] = mapped_column("Dimensione (KB)")

class Immagine(Base):
    __tablename__ = "Immagine"
    id: Mapped[str] = mapped_column(String(25), primary_key=True)
    Redacted: Mapped[int] = mapped_column("Redacted%",TINYINT)
    descrizione: Mapped[str] = mapped_column(String(400))
    size: Mapped[int] = mapped_column("Dimensione KB")
    Luogo: Mapped[str] = mapped_column("Nome Luogo",String(30))
    Stato: Mapped[str] = mapped_column("Paese",String(30))
    Comune: Mapped[str] = mapped_column(String(45))
    Via: Mapped[str] = mapped_column(String(45))
    cap: Mapped[int] = mapped_column("CAP")
    data: Mapped[datetime.datetime] = mapped_column(DateTime)

class Mail(Base): #comunicazione
    __tablename__ = "Mail"
    id: Mapped[str] = mapped_column("Mail ID",String(25), primary_key=True)
    Redacted: Mapped[int] = mapped_column("Redacted%",TINYINT)
    fil: Mapped[str] = mapped_column("File ID",String(20))
    enum: Mapped[TipoENUM] = mapped_column("Org o Pers")
    data: Mapped[datetime.datetime] = mapped_column(DateTime)
    persona: Mapped[str] = mapped_column("ID Mittente Persona",String(25))
    organizzazione: Mapped[str] = mapped_column("ID Mittente Organizzazione",String(30))
    descrizione: Mapped[str] = mapped_column(String(400))
    risposta: Mapped[str] = mapped_column("Risposta",String(25))
    tipo: Mapped[str] = mapped_column(String(10))
    indirizzo: Mapped[str] = mapped_column("Indirizzo Mittente",String(45))
    telefono: Mapped[str] = mapped_column("Num telefono mittente",String(20))
    oggetto: Mapped[str] = mapped_column("Oggetto",String(45))

class Destinatari(Base):
    __tablename__ = "Destinatari"
    num: Mapped[int] = mapped_column("Num Destinatario",primary_key= True)
    id: Mapped[str] = mapped_column("Mail ID",String(25), primary_key= True)
    indirizzo: Mapped[str] = mapped_column("Indirizzo mail destinatario",String(45))
    telefono: Mapped[str] = mapped_column("Num telefono destinatario",String(20))
    enum: Mapped[TipoENUM] = mapped_column("Org o Pers", primary_key= True)
    persona: Mapped[str] = mapped_column("ID Persona",String(25))
    organizzazione: Mapped[str] = mapped_column("ID Organizzazione",String(30))

class Soggetto(Base):
    __tablename__ = "Soggetto"
    num: Mapped[int] = mapped_column("N. rif del file",primary_key= True)
    id: Mapped[str] = mapped_column("File ID",String(20), primary_key= True)
    enum: Mapped[TipoENUM] = mapped_column("Org o Pers", primary_key= True)
    persona: Mapped[str] = mapped_column("ID Persona",String(25))
    organizzazione: Mapped[str] = mapped_column("ID Organizzazione",String(30))

class PersINimg(Base):
    __tablename__ = "Persona in immagine"
    id: Mapped[str] = mapped_column("ID Immagine",String(25), primary_key= True)
    pers: Mapped[str] = mapped_column("ID Persona",String(25), primary_key= True)
    fil: Mapped[str] = mapped_column("ID File",String(20))

class Lavoro(Base):
    __tablename__ = "Lavoro"
    num: Mapped[int] = mapped_column("Num. Lavoro",primary_key= True)
    id: Mapped[str] = mapped_column("ID Lavoratore",primary_key= True)
    salario: Mapped[int] = mapped_column("Salario")
    enum: Mapped[TipoENUM] = mapped_column("Org o Pers", primary_key= True)
    persona: Mapped[str] = mapped_column("Lavoraperpersona",String(25))
    organizzazione: Mapped[str] = mapped_column("Lavoraperorg",String(30))
    tipo: Mapped[str] = mapped_column("Tipo",String(45))



Inizializzazione del DB stesso

In [8]:
Base.metadata.create_all(bind=motore)

2026-05-11 17:06:39,363 INFO sqlalchemy.engine.Engine SELECT DATABASE()
2026-05-11 17:06:39,366 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,371 INFO sqlalchemy.engine.Engine SELECT @@sql_mode
2026-05-11 17:06:39,372 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,376 INFO sqlalchemy.engine.Engine SELECT @@lower_case_table_names


2026-05-11 17:06:39,377 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,382 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-11 17:06:39,383 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`Persona`
2026-05-11 17:06:39,385 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,392 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`Organizzazione`
2026-05-11 17:06:39,393 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,397 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`File`
2026-05-11 17:06:39,398 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,401 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`Immagine`
2026-05-11 17:06:39,402 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,405 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`Mail`
2026-05-11 17:06:39,406 INFO sqlalchemy.engine.Engine [raw sql] {}
2026-05-11 17:06:39,410 INFO sqlalchemy.engine.Engine DESCRIBE `mydb`.`Destinatari`
2026-05-11 17:06:39,412 INFO sqlalchemy.e

Creazione della sessione

In [9]:
Connessione = sessionmaker(bind=motore)
db = Connessione()



Esempi di query

In [ ]:
query1 = select(Persona)
tabquery = db.scalars(query1).all()
for riga in tabquery:
    print("Riga 1", )

2026-05-11 13:16:28,744 INFO sqlalchemy.engine.Engine SELECT `Persona`.`ID persona`, `Persona`.nome, `Persona`.cognome, `Persona`.`Secondo nome`, `Persona`.titolo, `Persona`.sesso, `Persona`.`Stato di residenza`, `Persona`.`Comune`, `Persona`.`Via`, `Persona`.`CAP`, `Persona`.`Tipo/Ruolo`, `Persona`.`Lavora per` 
FROM `Persona`
2026-05-11 13:16:28,745 INFO sqlalchemy.engine.Engine [cached since 111.3s ago] {}
Riga 1 CSFSVSVFSV Alonzo
Riga 1 PSCOP Patrick


In [12]:
query2 = select(func.contasoldi("PSCOP").label("tot"))
tabquery = db.execute(query2).all()
for riga in tabquery:
    print("Riga 1", riga.tot)

2026-05-11 17:11:15,272 INFO sqlalchemy.engine.Engine SELECT contasoldi(%(contasoldi_1)s) AS tot
2026-05-11 17:11:15,275 INFO sqlalchemy.engine.Engine [generated in 0.00334s] {'contasoldi_1': 'PSCOP'}
Riga 1 140000
